# Create Klingenstein-Simons Fellowship Awards in Neuroscience (FELLOWSHIP PATTERN, Method 6 Playwright)

The Esther A. & Joseph Klingenstein Fund's Klingenstein Fellowship Awards in Neuroscience (renamed Klingenstein-Simons Fellowship Awards in Neuroscience after the Simons Foundation joined as co-funder around 2010) is a US biomedical fellowship program supporting early-career investigators in basic or clinical neuroscience.

**First Method-6 (Playwright) ingest on the project.** klingenstein.org is Cloudflare-protected; plain HTTP requests return 403 even with browser-like User-Agent. The scraper uses Playwright + playwright-stealth via the new shared `scripts/local/_playwright_helper.py` module to bypass the JS challenge.

**Awarding body:** Esther A. and Joseph Klingenstein Fund - F4320306403 (US, DOI 10.13039/100001391)

**Source:**
- Current cohorts (2023, 2024, 2025) inline on `klingenstein.org/esther-a-joseph-klingenstein-fund/neuroscience/fellowship-programs/`
- Per-year archives 1981-2022 at `klingenstein.org/grantees/grantee/eajk-neuroscience-fellows/{YEAR}`

Years with no fellows announced (1982, 1984, 1986) are silently absent from the archive index. The current-cohort page exposes the available archive years as text-content `<a>` links; the scraper uses that as the authoritative year list rather than hardcoding.

**Schema choices:**
- One row per (fellow, year). `funder_award_id` = `eajk-{year}-{name-slug}`.
- `funder_scheme` reflects the program name as it appeared in the announcement year:
    - pre-2010 (265 rows): `Klingenstein Fellowship Awards in Neuroscience`
    - 2010-2025 (192 rows): `Klingenstein-Simons Fellowship Awards in Neuroscience`
- `funding_type` = `fellowship` uniformly.
- `amount` / `currency` ship NULL with **§6.7 amount-coverage WAIVED**: program-level amount has shifted over 45 years (and is currently published at USD 300,000 over 3 years per fellow, but not announced per-cohort historically). Same waiver pattern as HHMI (#44) / Damon Runyon (#73) / CIFAR (#79) / Packard (#95).
- `lead_investigator.affiliation.country` hardcoded to `US` — program eligibility limits to US institutions per the foundation's own page.
- `description` = research project title (the line below each fellow's name on post-~2005 pages). Older cohorts (1981-~2004) often list only Name + Institution; description is sparse there (~72% overall coverage).

**S3 location:** `s3a://openalex-ingest/awards/klingenstein_simons/klingenstein_simons_fellows.parquet`

**Provenance:** `klingenstein_simons` (verified count=0 on production).

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.klingenstein_simons_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/klingenstein_simons/klingenstein_simons_fellows.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.klingenstein_simons_raw;

## Step 1.5: Inspect raw + scheme/year/coverage

Expected: ~457 fellows, 1981-2025 (42 distinct years), 100% name, ~97% institution, ~72% description. Scheme split ~265 pre-Simons + ~192 post-Simons (transition year hardcoded at 2010).

In [ ]:
%sql
DESCRIBE openalex.awards.klingenstein_simons_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.klingenstein_simons_raw LIMIT 5;

In [ ]:
%sql
-- Per-(scheme, year) row counts.
SELECT scheme_label, year, COUNT(*) AS rows,
       COUNT(institution) AS has_inst,
       COUNT(research_title) AS has_title
FROM openalex.awards.klingenstein_simons_raw
GROUP BY scheme_label, year
ORDER BY year;

In [ ]:
%sql
-- Top institutions (sanity check parsing didn't bleed name into institution).
SELECT institution, COUNT(*) AS rows
FROM openalex.awards.klingenstein_simons_raw
WHERE institution IS NOT NULL
GROUP BY institution
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast - verify Esther & Joseph Klingenstein Fund funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306403;

## Step 2: Transform to award schema

Per-row `display_name` = `{scheme} - {fellow name} ({year})`. `description` = the research project title (NULL for older cohorts where the source doesn't publish project titles).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.klingenstein_simons_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306403  -- Esther A. and Joseph Klingenstein Fund
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(n.scheme_label, ' - ', n.name,
           CASE WHEN n.year IS NOT NULL THEN CONCAT(' (', n.year, ')') ELSE '' END) AS display_name,
    n.research_title AS description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'fellowship' AS funding_type,
    n.scheme_label AS funder_scheme,
    'klingenstein_simons' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.institution AS name,
                CAST('US' AS STRING) AS country,  -- program eligibility limits to US institutions per the foundation page
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.klingenstein_simons_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 147

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'klingenstein_simons' AND priority = 147;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    147 AS priority  -- Klingenstein-Simons Fellowship in Neuroscience
FROM openalex.awards.klingenstein_simons_awards;

## Step 6: Verification

§6.7 amount-coverage check **WAIVED** (same justification as HHMI #44 / Damon Runyon #73 / CIFAR #79 / Packard #95). All other completeness checks should be 95%+ on name, ~97% on institution, ~72% on description (older cohorts don't publish project titles).

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.klingenstein_simons_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.klingenstein_simons_awards;

In [ ]:
%sql
-- §6.3 Data completeness. pct_amount expected 0 (waiver).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    COUNT(start_year) AS has_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.klingenstein_simons_awards;

In [ ]:
%sql
-- §6.7 amount-waiver confirmation. Expect all NULL.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.klingenstein_simons_awards;

In [ ]:
%sql
-- Scheme split (pre- vs post-Simons collaboration).
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year
FROM openalex.awards.klingenstein_simons_awards
GROUP BY funder_scheme
ORDER BY min_year;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.klingenstein_simons_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS scheme, funding_type, start_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS institution
FROM openalex.awards.klingenstein_simons_awards
ORDER BY start_year DESC, family
LIMIT 10;